In [1]:
from neo4j import GraphDatabase
import pandas as pd
import json
from typing import List, Tuple, Dict
import logging

In [2]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Neo4j Connection
driver = None


connect_neo4j(
    uri="neo4j://127.0.0.1:7687",
    username="neo4j", 
    password="sakuni200211"
)


In [3]:
def connect_neo4j(uri: str, username: str, password: str):
    """
    Connect to Neo4j database
    
    Args:
        uri: Neo4j database URI (e.g., "bolt://localhost:7687")
        username: Database username (default: "neo4j")
        password: Database password
    """
    global driver
    driver = GraphDatabase.driver(uri, auth=(username, password))
    logger.info("✓ Connected to Neo4j database")



In [4]:
def close_connection():
    """Close the database connection"""
    global driver
    if driver:
        driver.close()
        logger.info("✓ Database connection closed")


In [5]:
def clear_database():
    """Clear all nodes and relationships (use with caution!)"""
    with driver.session() as session:
        session.run("MATCH (n) DETACH DELETE n")
        logger.info("✓ Database cleared")


In [6]:


def create_indexes():
    """Create indexes for better query performance"""
    with driver.session() as session:
        indexes = [
            "CREATE INDEX IF NOT EXISTS FOR (i:Ingredient) ON (i.name)",
            "CREATE INDEX IF NOT EXISTS FOR (n:Nutrient) ON (n.name)",
            "CREATE INDEX IF NOT EXISTS FOR (h:HealthOutcome) ON (h.name)",
            "CREATE INDEX IF NOT EXISTS FOR (f:FoodProduct) ON (f.name)",
            "CREATE INDEX IF NOT EXISTS FOR (d:Disease) ON (d.name)",
            "CREATE INDEX IF NOT EXISTS FOR (a:Allergen) ON (a.name)",
        ]
        
        for index_query in indexes:
            session.run(index_query)
        
        logger.info("✓ Indexes created successfully")



In [7]:
def load_triplets_from_list(triplets: List[Tuple[str, str, str]], batch_size: int = 1000):
    """
    Load triplets from a list of tuples
    
    Args:
        triplets: List of (subject, predicate, object) tuples
        batch_size: Number of triplets to process in each batch
    
    Example:
        triplets = [
            ("high_sodium", "increases_risk", "hypertension"),
            ("vitamin_c", "supports", "immune_system"),
            ("saturated_fat", "contributes_to", "obesity")
        ]
        load_triplets_from_list(triplets)
    """
    total = len(triplets)
    logger.info(f"Loading {total} triplets in batches of {batch_size}...")
    
    with driver.session() as session:
        for i in range(0, total, batch_size):
            batch = triplets[i:i + batch_size]
            
            query = """
            UNWIND $triplets AS triplet
            MERGE (s:Entity {name: triplet.subject})
            MERGE (o:Entity {name: triplet.object})
            MERGE (s)-[r:RELATES_TO {type: triplet.predicate}]->(o)
            """
            
            session.run(query, triplets=[
                {"subject": subj, "predicate": pred, "object": obj}
                for subj, pred, obj in batch
            ])
            
            logger.info(f"✓ Loaded {min(i + batch_size, total)}/{total} triplets")
    
    logger.info("✓ All triplets loaded successfully!")


In [8]:
def load_triplets_from_csv(csv_file="pubmed_triples.csv",
    subject_col="ingredient",
    predicate_col="relationship_type",
    object_col="disease",
    pmid_col="pmid",
    evidence_col="key_findings",
    batch_size: int = 1000):
    """
    Load triplets from CSV file
    
    Args:
        csv_file: Path to CSV file
        subject_col: Name of subject column
        predicate_col: Name of predicate column
        object_col: Name of object column
        batch_size: Number of triplets to process in each batch
    
    Example CSV format:
        subject,predicate,object
        high_sodium,increases_risk,hypertension
        vitamin_c,supports,immune_system
    """
    logger.info(f"Reading triplets from {csv_file}...")
    df = pd.read_csv(csv_file)
    
    triplets = [
        (row[subject_col], row[predicate_col], row[object_col])
        for _, row in df.iterrows()
    ]
    
    load_triplets_from_list(triplets, batch_size)


In [9]:
def load_triplets_with_labels(triplets: List[Tuple[str, str, str]], 
                              node_labels: Dict[str, str] = None,
                              batch_size: int = 1000):
    """
    Load triplets with custom node labels
    
    Args:
        triplets: List of (subject, predicate, object) tuples
        node_labels: Dict mapping node names to labels
        batch_size: Number of triplets to process in each batch
    
    Example:
        triplets = [("sodium", "increases_risk", "hypertension")]
        node_labels = {
            "sodium": "Nutrient",
            "hypertension": "Disease"
        }
        load_triplets_with_labels(triplets, node_labels)
    """
    node_labels = node_labels or {}
    total = len(triplets)
    logger.info(f"Loading {total} triplets with custom labels...")
    
    with driver.session() as session:
        for i in range(0, total, batch_size):
            batch = triplets[i:i + batch_size]
            
            for subj, pred, obj in batch:
                subj_label = node_labels.get(subj, "Entity")
                obj_label = node_labels.get(obj, "Entity")
                
                query = f"""
                MERGE (s:{subj_label} {{name: $subject}})
                MERGE (o:{obj_label} {{name: $object}})
                MERGE (s)-[r:RELATES_TO {{type: $predicate}}]->(o)
                """
                
                session.run(query, subject=subj, predicate=pred, object=obj)
            
            logger.info(f"✓ Loaded {min(i + batch_size, total)}/{total} triplets")
    
    logger.info("✓ All triplets loaded with custom labels!")


In [10]:
def query_triplets(subject: str = None, predicate: str = None, object_name: str = None):
    """
    Query triplets from the database
    
    Args:
        subject: Filter by subject name
        predicate: Filter by relationship type
        object_name: Filter by object name
    
    Returns:
        List of matching triplets
    """
    with driver.session() as session:
        conditions = []
        params = {}
        
        if subject:
            conditions.append("s.name = $subject")
            params["subject"] = subject
        if predicate:
            conditions.append("r.type = $predicate")
            params["predicate"] = predicate
        if object_name:
            conditions.append("o.name = $object")
            params["object"] = object_name
        
        where_clause = " AND ".join(conditions) if conditions else "true"
        
        query = f"""
        MATCH (s)-[r:RELATES_TO]->(o)
        WHERE {where_clause}
        RETURN s.name AS subject, r.type AS predicate, o.name AS object
        LIMIT 100
        """
        
        result = session.run(query, **params)
        triplets = [(record["subject"], record["predicate"], record["object"]) 
                   for record in result]
        
        return triplets


In [11]:
def get_database_stats():
    """Get statistics about the loaded knowledge graph"""
    with driver.session() as session:
        node_count = session.run("MATCH (n) RETURN count(n) AS count").single()["count"]
        rel_count = session.run("MATCH ()-[r]->() RETURN count(r) AS count").single()["count"]
        
        logger.info(f"Database Stats: {node_count} nodes, {rel_count} relationships")
        return {"nodes": node_count, "relationships": rel_count}


In [12]:
if __name__ == "__main__":
    # 1. Connect to Neo4j
    connect_neo4j(
        uri="neo4j://127.0.0.1:7687",
        username="neo4j",
        password="sakuni200211"
    )
    
    # 2. Optional: Clear existing data
    # clear_database()
    
    # 3. Create indexes for performance
    create_indexes()
    
    # 4. Load triplets (choose one method)
    
    # Method 1: From list
    triplets = [
        ("high_sodium", "increases_risk", "hypertension"),
        ("saturated_fat", "contributes_to", "obesity"),
        ("vitamin_c", "supports", "immune_system"),
        ("trans_fat", "increases_risk", "cardiovascular_disease"),
        ("fiber", "reduces_risk", "diabetes")
    ]
    load_triplets_from_list(triplets)
    
    # Method 2: From CSV
    load_triplets_from_csv("pubmed_triples.csv")
    
    # Method 3: From JSON
    # load_triplets_from_json("triplets.json")
    
    # Method 4: With custom labels
    # node_labels = {
    #     "high_sodium": "Nutrient",
    #     "hypertension": "Disease",
    #     "vitamin_c": "Nutrient",
    #     "immune_system": "HealthOutcome"
    # }
    # load_triplets_with_labels(triplets, node_labels)
    
    # 5. Get database statistics
    get_database_stats()
    
    # 6. Query some triplets
    results = query_triplets(predicate="increases_risk")
    print(f"\nFound {len(results)} triplets with 'increases_risk' relationship:")
    for s, p, o in results[:5]:
        print(f"  {s} -> {p} -> {o}")
    
    # 7. Close connection
    close_connection()

INFO:__main__:✓ Connected to Neo4j database
INFO:neo4j.notifications:Received notification from DBMS server: <GqlStatusObject gql_status='00NA0', status_description="note: successful completion - index or constraint already exists. The command 'CREATE RANGE INDEX IF NOT EXISTS FOR (e:Ingredient) ON (e.name)' has no effect. The index or constraint specified by 'RANGE INDEX index_ec1ff354 FOR (e:Ingredient) ON (e.name)' already exists.", position=None, raw_classification='SCHEMA', classification=<NotificationClassification.SCHEMA: 'SCHEMA'>, raw_severity='INFORMATION', severity=<NotificationSeverity.INFORMATION: 'INFORMATION'>, diagnostic_record={'_classification': 'SCHEMA', '_severity': 'INFORMATION', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CREATE INDEX IF NOT EXISTS FOR (i:Ingredient) ON (i.name)'
INFO:neo4j.notifications:Received notification from DBMS server: <GqlStatusObject gql_status='00NA0', status_description="note: successful completion - ind


Found 2 triplets with 'increases_risk' relationship:
  high_sodium -> increases_risk -> hypertension
  trans_fat -> increases_risk -> cardiovascular_disease
